## Librerias

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler


## Datos

In [3]:
df = pd.read_csv('dfSegmentado.csv')

## Procesamiento

In [4]:
target = "Segmento"
X = df.drop(columns=["Segmento"])  # variables predictoras
y = df["Segmento"]                 # variable objetivo

X = X.drop(columns=["Identificador"], errors="ignore")

In [5]:
numericas = X.select_dtypes(include=["int64", "float64"]).columns
categoricas = X.select_dtypes(include=["object", "category"]).columns

In [6]:
X = pd.get_dummies(X, columns=categoricas, drop_first=True)

In [7]:
scaler = StandardScaler()
X[numericas] = scaler.fit_transform(X[numericas])

## Modelo - Arbol de decision

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
dt_model = DecisionTreeClassifier(
    max_depth=5,        # control overfitting
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Accuracy: 0.48451053283767037
              precision    recall  f1-score   support

           A       0.39      0.28      0.33       394
           B       0.32      0.33      0.32       372
           C       0.55      0.56      0.55       394
           D       0.61      0.72      0.66       454

    accuracy                           0.48      1614
   macro avg       0.47      0.47      0.47      1614
weighted avg       0.47      0.48      0.48      1614



## Modelo - Random Forest

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.5241635687732342
              precision    recall  f1-score   support

           A       0.42      0.44      0.43       394
           B       0.39      0.28      0.33       372
           C       0.57      0.58      0.58       394
           D       0.63      0.74      0.68       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.51      1614
weighted avg       0.51      0.52      0.51      1614



## Comparacion 

In [11]:
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Decision Tree Accuracy: 0.48451053283767037
Random Forest Accuracy: 0.5241635687732342


## Optimizacion Random Forest

In [18]:
features_clave = [
    "Edad",
    "Tamano_Familiar",
    "Experiencia_Laboral",
    "Puntuacion_Gasto_Low",
    "Graduado_Yes",
    "Estado_Civil_Yes"
]

X_reducido = X[features_clave]

In [19]:
df["edad_productiva"] = (df["Edad"] >= 25) & (df["Edad"] <= 55)
df["edad_productiva"] = df["edad_productiva"].astype(int)

In [20]:
df["ratio_exp_edad"] = df["Experiencia_Laboral"] / (df["Edad"] + 1)

In [21]:
df["carga_familiar"] = df["Tamano_Familiar"] / (df["Edad"] + 1)

In [23]:
df["gasto_alto"] = (df["Puntuacion_Gasto"] == "High").astype(int)

In [24]:
# Reentrenar modelos con nuevas características
X = df[[
    "Edad",
    "Experiencia_Laboral",
    "Tamano_Familiar",
    "ratio_exp_edad",
    "carga_familiar",
    "edad_productiva",
    "gasto_alto"
]]

y = df["Segmento"]

In [25]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=5,
    random_state=42
)

In [26]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

In [30]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.442998760842627
              precision    recall  f1-score   support

           A       0.35      0.35      0.35       394
           B       0.29      0.11      0.16       372
           C       0.41      0.58      0.48       394
           D       0.58      0.68      0.63       454

    accuracy                           0.44      1614
   macro avg       0.41      0.43      0.40      1614
weighted avg       0.42      0.44      0.42      1614

